In [ ]:
import sys
print("The python version is: " + sys.version)

# AdvTG — end-to-end pipeline

Adversarial HTTP traffic generation vs. DL malicious-traffic detectors, run stage by stage:

1. **Dataset** → 2. **Detectors** (token + image) → 3. **LLM finetune** (QLoRA) → 4. **PPO** adversarial generation

**Where this runs:** a **Kaggle notebook** (GPU). The cells run on Kaggle's VM, not your laptop.

- Notebook ▸ Settings: **Accelerator = GPU T4 x2** (avoid P100: current torch/bitsandbytes builds don't support it), **Internet = On** (needed for git clone, pip, apt, and HF downloads).
- Runs on Kaggle's default Python. The stack is modern and unpinned, so you don't need a special runtime, a torch downgrade, or kernel restarts.
- The VM only has what's in the **git remote**. **Push your branch first**; the Setup cell clones it into `/kaggle/working/AdvTG` (and `git pull`s on re-runs). Local edits do **not** sync to the VM.
- Storage: the repo, `dataset/` and `model/` live under **`/kaggle/working`**, which is kept as notebook output when you *Save Version* (20 GB limit). HF model weights and the CIC-IDS2017 captures are cached outside it, so they don't count against that limit.
- **`HF_TOKEN`** under Add-ons ▸ Secrets: optional for reading, required (with **write** access) for the *Publish* cell.
- Only GPU 0 is used (`CUDA_VISIBLE_DEVICES=0`). Everything fits on one T4, and this avoids multi-GPU surprises.

**Data:** Stage 1 and Stage 3 both read the real CIC-IDS2017 captures from the `bencorn/CICIDS2017` mirror, through the shared *PCAP helpers* cell. Stage 1 builds the detectors' dataset from an attack day, labelled per request by CIC's attack hosts; Stage 3 fine-tunes the LLM on benign Monday traffic.

**Build once, reuse:** after Stage 2, the *Publish* cell uploads the parsed dataset and the trained detectors to Hugging Face. On any later run, the *Pull* cell restores both in a minute, and Stages 1–2 can be skipped.

**Each stage installs its own deps** at the top of its first cell (there's no central install step), so you can run any stage on its own. `requirements.txt` still lists the full stack for venv/Docker use. Run **Setup + Config + PCAP helpers** once, then run the stages in order.

In [ ]:
import os, sys, subprocess

# Kaggle T4 x2: pin to one GPU (must be set before torch initialises CUDA).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# HF token from Kaggle Secrets (Add-ons ▸ Secrets ▸ HF_TOKEN) — optional.
if "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass   # no secret attached -> unauthenticated HF downloads

# Setup as a function. Light + idempotent (git pull + chdir + sys.path — NO torch import),
# so it's safe to call at the top of any stage via prepare(). Run this cell once to define it.
GIT_URL = "https://github.com/TejaswiMN/AdvTG.git"
BRANCH  = "LLM-Finetune"
WORK    = "/kaggle/working"            # persisted as notebook output on Save Version

def setup():
    """Clone/pull the repo, cd into it, make it importable. Sets global REPO."""
    global REPO
    if os.path.isfile(os.path.join(os.getcwd(), "gen_synthetic_data.py")):
        REPO = os.getcwd()                                   # already inside the repo
    else:
        # clone target on the hosted VM (Kaggle)
        REPO = os.path.join(WORK, "AdvTG")
        if os.path.isdir(os.path.join(REPO, ".git")):
            subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=False)
            subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=False)
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
        else:
            subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO], check=True)
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)                             # make the DL package importable
    return REPO

setup()
print("repo:", REPO, "| has gen_synthetic_data.py:", os.path.isfile("gen_synthetic_data.py"))

## Config — the settings live in `config()`

To change a setting (dataset source, sizes, epochs…), edit the value **inside the `config()` function** below and re-run this cell. Everything downstream reads `dataset/train_data2.json`.

`setup()` and `config()` are light, torch-free helpers; `prepare()` runs both and is called at the top of every stage — so each stage is self-contained and you never have to scroll back up to re-run Setup/Config.

In [ ]:
import os

# Config as a function (light + idempotent, no torch import). prepare() = setup() + config(),
# called at the top of every stage so each stage is self-contained. To change a setting,
# edit the value inside config() and re-run this cell.
def config():
    """Define dataset/training settings + paths (needs REPO from setup())."""
    global DATASET_SOURCE, N_TRAIN, N_TEST, MALICIOUS_RATIO
    global HF_DATASET_REPO, HF_LLM_PCAPS, HF_LLM_MAX_REQUESTS
    global HF_DETECTOR_PCAPS, DETECTOR_MAX_PER_CLASS, ATTACK_IPS
    global HF_USER, HF_TRAFFIC_REPO, HF_DETECTOR_REPO, HF_PRIVATE
    global MAX_LENGTH, BATCH_SIZE, NUM_EPOCHS, TRAIN_BERT
    global DATA_DIR, MODEL_DIR, TRAIN_JSON, TEST_JSON

    # --- dataset settings -----------------------------------------------------
    DATASET_SOURCE  = "cicids2017"      # "synthetic" | "cicids2017" | "hub"
                                        # "hub" = pull the dataset a previous run published (no parsing)
    N_TRAIN         = 20000             # synthetic only
    N_TEST          = 4000
    MALICIOUS_RATIO = 0.35
    HF_DATASET_REPO = "bencorn/CICIDS2017"      # CIC-IDS2017 pcap mirror, used by Stage 1 and Stage 3
    # Stage 1 (detectors) parses these captures. The label is per capture: "Benign" / "Malicious"
    # labels the whole file; "by-host" labels each request by whether it touches one of ATTACK_IPS,
    # so a single attack day gives both classes.
    HF_DETECTOR_PCAPS = [("Thursday-WorkingHours.pcap", "by-host")]   # web attacks: brute force, XSS, SQLi
    DETECTOR_MAX_PER_CLASS = 10000      # reservoir-sampled, per capture per label
    # CIC-IDS2017's documented attack hosts: the Kali box, the Windows attackers, the firewall's outside
    # address, and 172.16.0.1, the inside address all their traffic is NATed through. 192.168.10.8 is
    # left out on purpose: it is the infiltration victim, but it is also an ordinary user browsing all
    # day, so tagging its HTTP as malicious would mislabel mostly benign traffic.
    ATTACK_IPS = {"205.174.165.73", "205.174.165.80", "205.174.165.69",
                  "205.174.165.70", "205.174.165.71", "172.16.0.1"}
    # Stage 3 (LLM) downloads exactly this one PCAP from Hugging Face.
    HF_LLM_PCAPS = [("Monday-WorkingHours.pcap", "Benign")]
    HF_LLM_MAX_REQUESTS = 10000
    # --- publishing: needs an HF_TOKEN secret with WRITE access --------------
    HF_USER          = "your-hf-username"
    HF_TRAFFIC_REPO  = f"{HF_USER}/advtg-cicids2017-http"   # parsed train/test JSON (dataset repo)
    HF_DETECTOR_REPO = f"{HF_USER}/advtg-detectors"         # trained detector weights (model repo)
    HF_PRIVATE       = True
    # detector training
    MAX_LENGTH = 512
    BATCH_SIZE = 16
    NUM_EPOCHS = 2
    TRAIN_BERT = False
    # paths
    DATA_DIR   = os.path.join(REPO, "dataset")
    MODEL_DIR  = os.path.join(REPO, "model")
    TRAIN_JSON = os.path.join(DATA_DIR, "train_data2.json")
    TEST_JSON  = os.path.join(DATA_DIR, "test2.json")
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

def prepare():
    """setup() + config() — call once at the top of any stage."""
    setup(); config()

config()
print("dataset source:", DATASET_SOURCE, "| model dir:", MODEL_DIR)
print("detector PCAPs:", HF_DETECTOR_PCAPS, "| max per class:", DETECTOR_MAX_PER_CLASS)
print("LLM PCAPs:", HF_LLM_PCAPS, "| max requests:", HF_LLM_MAX_REQUESTS)

In [ ]:
# PCAP helpers (torch-free), shared by Stage 1 (detector data) and Stage 3 (LLM data): install
# tshark, fetch a capture from the HF mirror, stream the HTTP requests out of it. Run once, like
# Setup and Config.
import collections, contextlib, os, random, shutil, subprocess, tempfile
from huggingface_hub import hf_hub_download

# One -e field per column. Headers are asked for one by one rather than via http.request.line,
# which comes back as comma-joined values that turn ambiguous as soon as a header holds a comma.
REQUEST_FIELDS = ["ip.src", "ip.dst", "http.request.method", "http.request.uri",
                  "http.request.version", "http.file_data"]
HEADER_FIELDS = {"http.host": "Host", "http.user_agent": "User-Agent", "http.accept": "Accept",
                 "http.accept_language": "Accept-Language", "http.accept_encoding": "Accept-Encoding",
                 "http.referer": "Referer", "http.cookie": "Cookie",
                 "http.content_type": "Content-Type", "http.connection": "Connection"}

def ensure_tshark():
    """Kaggle's image has no tshark. Install it once; the debconf preseed stops apt from hanging."""
    if shutil.which("tshark"):
        return
    subprocess.run("echo 'wireshark-common wireshark-common/install-setuid boolean false' "
                   "| debconf-set-selections", shell=True, check=False)
    subprocess.run("DEBIAN_FRONTEND=noninteractive apt-get -qq update && "
                   "DEBIAN_FRONTEND=noninteractive apt-get -qq install -y tshark",
                   shell=True, check=True)

_KNOWN_FIELDS = None
def usable_fields(fields):
    """Drop -e fields this tshark build doesn't know — a single unknown one aborts it outright."""
    global _KNOWN_FIELDS
    if _KNOWN_FIELDS is None:
        listing = subprocess.run(["tshark", "-G", "fields"], capture_output=True, text=True).stdout
        _KNOWN_FIELDS = {p[2] for p in (l.split("\t") for l in listing.splitlines())
                         if len(p) > 2 and p[0] == "F"}
    missing = [f for f in fields if _KNOWN_FIELDS and f not in _KNOWN_FIELDS]
    if missing:
        print("  tshark doesn't know", missing, "- continuing without them")
    return [f for f in fields if f not in missing]

def download_pcap(filename):
    """The HF mirror keeps some captures at the repo root and others under pcaps/."""
    try:
        return hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename=filename)
    except Exception:
        return hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                               filename="pcaps/" + filename)

def iter_http_requests(path, source):
    """Yield (src_ip, dst_ip, record) for every HTTP request tshark reassembles from a capture."""
    ensure_tshark()
    fields = usable_fields(REQUEST_FIELDS + list(HEADER_FIELDS))
    cmd = ["tshark", "-r", path,
           "-o", "tcp.desegment_tcp_streams:TRUE",
           "-o", "http.desegment_headers:TRUE",
           "-o", "http.desegment_body:TRUE",
           "-Y", "http.request", "-T", "fields",
           "-E", "separator=/t",        # tshark keeps only one character; "/t" is its spelling of TAB
           "-E", "occurrence=f"]
    for f in fields:
        cmd += ["-e", f]
    stderr_file = tempfile.TemporaryFile()   # a file, not a pipe: a chatty stderr can't stall the read
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=stderr_file, text=True,
                            errors="replace", bufsize=1 << 20)
    finished = False
    try:
        for line in proc.stdout:
            values = line.rstrip("\r\n").split("\t")
            row = dict(zip(fields, values + [""] * (len(fields) - len(values))))
            method, uri = row.get("http.request.method", ""), row.get("http.request.uri", "")
            if not method or not uri:
                continue
            body = row.get("http.file_data", "")
            if body and ":" in body and " " not in body:          # binary bodies arrive as hex
                try:
                    body = bytes.fromhex(body.replace(":", "")).decode("utf-8", "replace")
                except ValueError:
                    pass
            yield row.get("ip.src", ""), row.get("ip.dst", ""), {
                "Request Line": "%s %s %s" % (method, uri, row.get("http.request.version") or "HTTP/1.1"),
                "Request Headers": {name: row[f] for f, name in HEADER_FIELDS.items() if row.get(f)},
                "Request Body": body,
                "Source": source,
            }
        finished = True
    finally:
        if not finished:                # the caller stopped early: that's expected, not an error
            proc.kill()
        proc.stdout.close()
        code = proc.wait()
        stderr_file.seek(0)
        message = stderr_file.read().decode("utf-8", "replace").strip()
        stderr_file.close()
        if finished and code != 0:
            raise RuntimeError(f"tshark exited {code} on {path}: {message[-2000:] or '(no stderr)'}")

def collect_requests(pcaps, per_class, full_scan=True, seed=42):
    """Deduplicated HTTP requests from each (capture, label) pair, capped at per_class per label.

    "Benign" / "Malicious" label a whole capture; "by-host" labels each request by whether either
    endpoint is in ATTACK_IPS. full_scan=True reservoir-samples across the entire capture, because
    attacks run at fixed times of day and the first N requests would miss most of them. False stops
    as soon as the caps fill, which is all an all-benign capture needs (Stage 3).
    """
    rng, seen, out = random.Random(seed), set(), []
    for filename, how in pcaps:
        if how not in ("Benign", "Malicious", "by-host"):
            raise ValueError(f"{filename}: label must be 'Benign', 'Malicious' or 'by-host', not {how!r}")
        needed = ["Benign", "Malicious"] if how == "by-host" else [how]
        path = download_pcap(filename)
        print(f"{filename}: {os.path.getsize(path)/1e9:.1f} GB, labels: {how}", flush=True)
        pools, counts = {"Benign": [], "Malicious": []}, collections.Counter()
        with contextlib.closing(iter_http_requests(path, filename)) as rows:
            for src, dst, rec in rows:
                label = (("Malicious" if src in ATTACK_IPS or dst in ATTACK_IPS else "Benign")
                         if how == "by-host" else how)
                key = hash((rec["Request Line"], tuple(sorted(rec["Request Headers"].items())),
                            rec["Request Body"]))
                if key in seen:                 # floods and brute force repeat one request thousands of times
                    counts["duplicate"] += 1
                    continue
                seen.add(key)
                counts[label] += 1
                pool = pools[label]
                if len(pool) < per_class:
                    pool.append(dict(rec, Label=label))
                elif full_scan:                 # reservoir: uniform over the whole capture
                    j = rng.randrange(counts[label])
                    if j < per_class:
                        pool[j] = dict(rec, Label=label)
                elif all(len(pools[l]) >= per_class for l in needed):
                    break
                if (counts["Benign"] + counts["Malicious"]) % 100000 == 0:
                    print(f"    {dict(counts)}", flush=True)
        print(f"  unique requests seen {dict(counts)} -> kept "
              f"{ {l: len(p) for l, p in pools.items() if p} }", flush=True)
        out += pools["Benign"] + pools["Malicious"]
    return out

print("PCAP helpers ready: ensure_tshark, download_pcap, iter_http_requests, collect_requests")

## Stage 1 — Dataset

Produces `dataset/train_data2.json` (+ `test2.json`) — the single file every downstream stage reads.

- **`"cicids2017"`** — real HTTP requests parsed out of the CIC-IDS2017 captures in `HF_DETECTOR_PCAPS`,
  pulled from the `bencorn/CICIDS2017` mirror with the same tshark path Stage 3 uses. A `"by-host"` capture
  is labelled per request by CIC's documented attack hosts (`ATTACK_IPS`), so one attack day gives both
  classes. Requests are deduplicated, reservoir-sampled per label across the whole capture (attacks run at
  set times of day, so the first N requests would miss most of them), then split 80/20. Thursday takes
  roughly 10–25 minutes to download and scan.
- **`"hub"`** — the dataset a previous run published with the *Publish* cell after Stage 2. No parsing.
- **`"synthetic"`** — the repo's generator, for a quick smoke test.

To skip Stages 1–2 entirely on later runs, use the *Pull* cell after Stage 2 instead.

In [ ]:
import collections, json, os, random, shutil
prepare()   # re-establish repo + config, so this stage runs standalone

if DATASET_SOURCE == "synthetic":
    !python gen_synthetic_data.py --n {N_TRAIN} --test-n {N_TEST} \
        --malicious-ratio {MALICIOUS_RATIO} --out "{TRAIN_JSON}" --test-out "{TEST_JSON}"

elif DATASET_SOURCE == "cicids2017":
    # real HTTP requests from the CIC-IDS2017 captures (helpers: the PCAP helpers cell above)
    records = collect_requests(HF_DETECTOR_PCAPS, DETECTOR_MAX_PER_CLASS, full_scan=True)
    labels = collections.Counter(r["Label"] for r in records)
    if len(labels) < 2:
        raise RuntimeError(f"only {dict(labels)} came out of {HF_DETECTOR_PCAPS} — the detectors need "
                           "both classes. Add a capture with attack traffic, or check ATTACK_IPS.")
    # stratified 80/20 split -> train_data2.json / test2.json
    rng, train, test = random.Random(42), [], []
    for label in labels:
        group = [r for r in records if r["Label"] == label]
        rng.shuffle(group)
        cut = int(len(group) * 0.2)
        test += group[:cut]
        train += group[cut:]
    rng.shuffle(train); rng.shuffle(test)
    json.dump(train, open(TRAIN_JSON, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(test,  open(TEST_JSON,  "w", encoding="utf-8"), ensure_ascii=False)
    print(f"train {len(train):,} | test {len(test):,} | {dict(labels)} -> {DATA_DIR}")

elif DATASET_SOURCE == "hub":
    # the dataset a previous run published (see the Publish cell after Stage 2)
    from huggingface_hub import snapshot_download
    d = snapshot_download(HF_TRAFFIC_REPO, repo_type="dataset", token=os.environ.get("HF_TOKEN"))
    for name, dest in [("train_data2.json", TRAIN_JSON), ("test2.json", TEST_JSON)]:
        shutil.copy(os.path.join(d, name), dest)
    print("pulled", HF_TRAFFIC_REPO, "->", DATA_DIR)

else:
    raise ValueError("DATASET_SOURCE must be 'synthetic', 'cicids2017' or 'hub'")

In [ ]:
import json, collections
recs = json.load(open(TRAIN_JSON, encoding="utf-8"))
print(len(recs), "records", dict(collections.Counter(r["Label"] for r in recs)))
r = recs[0]
print("\n" + r["Request Line"])
for k, v in list(r["Request Headers"].items())[:4]:
    print(f"  {k}: {v}")
print("  ->", r["Label"], "| source:", r["Source"])

## Stage 2 — Detector training

Trains the token-level (TextCNN / CNN-LSTM / DNN) and image detectors, then writes the `model_configs` pickles the PPO stage attacks. Runs on CPU or GPU.

In [ ]:
# token-level detectors (TextCNN / CNN-LSTM / DNN), sharing the BERT tokenizer's vocab
prepare()   # re-establish repo + config (must run before the DL imports below)
# Stage 2 deps (torch/numpy come from the runtime):
!pip -q install "transformers>=4.46" "datasets>=2.20" "scikit-learn>=1.3"
import os, torch
import DL.training as _T
from transformers import TrainingArguments, AutoTokenizer
from DL.data_processing import load_data, prepare_dataset
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.training import train_custom_model, train_transformer_model

_T.MODEL_PATH = MODEL_DIR          # repo hardcodes ./models/; redirect saves under model/
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
data = load_data(TRAIN_JSON)
train_ds, val_ds, test_ds = prepare_dataset(data, tokenizer, MAX_LENGTH)

vocab_size, embed_size, num_classes = len(tokenizer.vocab), 128, 2
os.makedirs(os.path.join(MODEL_DIR, "custom_models"), exist_ok=True)
args = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "custom_models"),
                         per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                         learning_rate=2e-5, num_train_epochs=NUM_EPOCHS, report_to="none")

for name, model in {
        "textcnn":  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "cnn_lstm": CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "dnn":      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)}.items():
    print("training", name)
    train_custom_model(model, name, train_ds, val_ds, args)

if TRAIN_BERT:
    bargs = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "bert"),
                              evaluation_strategy="epoch", learning_rate=2e-5,
                              per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                              num_train_epochs=NUM_EPOCHS, weight_decay=0.01, save_strategy="epoch",
                              load_best_model_at_end=True, report_to="none")
    train_transformer_model("bert", TOKENIZER_NAME, train_ds, val_ds, bargs)

In [ ]:
# image-based detectors: each request rendered as a 28x28 byte image (ord(c) % 128)
import numpy as np, torch, os
from torch.utils.data import DataLoader, TensorDataset
from DL.data_processing import json_to_string
from DL.image_models import ImageCNN, ImageMLP

IMG = (28, 28)
def to_image(it):
    text = it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
    v = [ord(c) % 128 for c in text][:IMG[0] * IMG[1]]
    v += [0] * (IMG[0] * IMG[1] - len(v))
    return np.array(v, dtype=np.float32).reshape(IMG)

X = torch.tensor(np.stack([to_image(r) for r in data]))
y = torch.tensor([1 if r["Label"] == "Malicious" else 0 for r in data], dtype=torch.long)
k = int(len(X) * 0.9)
loader = DataLoader(TensorDataset(X[:k], y[:k]), batch_size=64, shuffle=True)

for name, m in {"imagecnn": ImageCNN(), "imagemlp": ImageMLP()}.items():
    m.to(device); opt = torch.optim.Adam(m.parameters(), 1e-3); lf = torch.nn.CrossEntropyLoss()
    for _ in range(int(NUM_EPOCHS)):
        m.train()
        for xb, yb in loader:
            loss = lf(m(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(X[k:].to(device)).argmax(1).cpu() == y[k:]).float().mean().item()
    path = os.path.join(MODEL_DIR, "custom_models", name + ".bin")
    torch.save(m.state_dict(), path)
    print(f"{name}: val acc {acc:.3f} -> {path}")

In [ ]:
import pickle, os
from transformers import AutoTokenizer
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.image_models import ImageCNN, ImageMLP

cm = os.path.join(MODEL_DIR, "custom_models")

def cfg(name, cls):
    return {"type": "custom", "name": name, "path": os.path.join(cm, name + ".bin"), "class": cls}

text_configs = [
    cfg("textcnn",  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("cnn_lstm", CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("dnn",      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
]
image_configs = [cfg("imagecnn", ImageCNN()), cfg("imagemlp", ImageMLP())]

pickle.dump(text_configs,  open(os.path.join(MODEL_DIR, "model_configs.pkl"), "wb"))
pickle.dump(image_configs, open(os.path.join(MODEL_DIR, "imgae_model_configs.pkl"), "wb"))

# PPO's Text reward re-tokenises responses with this — save it even when BERT is skipped.
AutoTokenizer.from_pretrained(TOKENIZER_NAME).save_pretrained(os.path.join(MODEL_DIR, "bert"))
print("wrote model_configs.pkl, imgae_model_configs.pkl, model/bert/ tokenizer")

## Publish — dataset + detectors to Hugging Face

Uploads what Stages 1–2 produced, so no later run has to rebuild them: `train_data2.json` / `test2.json` to
the dataset repo `HF_TRAFFIC_REPO`, and the five detectors' weights, the BERT tokenizer and a small manifest
to the model repo `HF_DETECTOR_REPO`. Run it in the same session, straight after Stage 2.

The `model_configs` pickles are **not** uploaded: they hold live model objects and absolute paths, so the
*Pull* cell rebuilds them on whatever machine pulls the detectors. Needs an `HF_TOKEN` secret with **write**
access, and `HF_USER` set in `config()`. Re-running updates the same repos in place; older versions stay in
their history.

In [ ]:
import datetime, json, os
prepare()   # needs vocab_size / embed_size / num_classes / TOKENIZER_NAME from the Stage 2 cells above
from huggingface_hub import HfApi

assert os.environ.get("HF_TOKEN"), "add an HF_TOKEN secret with write access (Add-ons ▸ Secrets)"
assert HF_USER != "your-hf-username", "set HF_USER in config() first"
api = HfApi(token=os.environ["HF_TOKEN"])

# 1) the parsed requests — Stage 3/4 read dataset/*.json too, so they travel with the detectors
api.create_repo(HF_TRAFFIC_REPO, repo_type="dataset", private=HF_PRIVATE, exist_ok=True)
for path in (TRAIN_JSON, TEST_JSON):
    api.upload_file(path_or_fileobj=path, path_in_repo=os.path.basename(path),
                    repo_id=HF_TRAFFIC_REPO, repo_type="dataset")

# 2) the detectors: weights, tokenizer, and what the Pull cell needs to rebuild the pickles
manifest = {
    "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "dataset": {"source": DATASET_SOURCE, "pcaps": HF_DETECTOR_PCAPS,
                "attack_ips": sorted(ATTACK_IPS), "hub_repo": HF_TRAFFIC_REPO},
    "tokenizer": TOKENIZER_NAME, "vocab_size": vocab_size, "embed_size": embed_size,
    "num_classes": num_classes, "max_length": MAX_LENGTH,
    "text_models": ["textcnn", "cnn_lstm", "dnn"], "image_models": ["imagecnn", "imagemlp"],
}
json.dump(manifest, open(os.path.join(MODEL_DIR, "detectors_manifest.json"), "w"), indent=2)
api.create_repo(HF_DETECTOR_REPO, repo_type="model", private=HF_PRIVATE, exist_ok=True)
api.upload_folder(folder_path=MODEL_DIR, repo_id=HF_DETECTOR_REPO,
                  allow_patterns=["custom_models/*.bin", "bert/*", "detectors_manifest.json"],
                  commit_message=f"detectors trained on {DATASET_SOURCE} {HF_DETECTOR_PCAPS}")
print("dataset   ->", f"https://huggingface.co/datasets/{HF_TRAFFIC_REPO}")
print("detectors ->", f"https://huggingface.co/{HF_DETECTOR_REPO}")

## Pull — skip Stages 1–2 on later runs

Run this **instead of** Stage 1 and Stage 2 (after Setup + Config). It restores `dataset/*.json`,
`model/custom_models/*.bin` and `model/bert/` from the repos the *Publish* cell wrote, then rebuilds
`model_configs.pkl` / `imgae_model_configs.pkl` locally. Stage 3 and Stage 4 then run as if Stages 1–2 had
just finished. A private repo needs an `HF_TOKEN` with read access; to use a teammate's detectors, set
`HF_USER` to their username.

In [ ]:
import json, os, pickle, shutil
prepare()
from huggingface_hub import snapshot_download

token = os.environ.get("HF_TOKEN")
d = snapshot_download(HF_TRAFFIC_REPO, repo_type="dataset", token=token)
for name in ("train_data2.json", "test2.json"):
    shutil.copy(os.path.join(d, name), os.path.join(DATA_DIR, name))
snapshot_download(HF_DETECTOR_REPO, token=token, local_dir=MODEL_DIR,
                  allow_patterns=["custom_models/*.bin", "bert/*", "detectors_manifest.json"])

# rebuild the pickles RL-Adv opens — same shape as the Stage 2 cell writes
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.image_models import ImageCNN, ImageMLP
m = json.load(open(os.path.join(MODEL_DIR, "detectors_manifest.json")))
cm = os.path.join(MODEL_DIR, "custom_models")

def cfg(name, obj):
    return {"type": "custom", "name": name, "path": os.path.join(cm, name + ".bin"), "class": obj}

dims = (m["vocab_size"], m["embed_size"], m["num_classes"], m["max_length"])
text_configs = [cfg("textcnn",  TextCNNClassifier(*dims)),
                cfg("cnn_lstm", CNNLSTMClassifier(*dims)),
                cfg("dnn",      DNNClassifier(*dims))]
image_configs = [cfg("imagecnn", ImageCNN()), cfg("imagemlp", ImageMLP())]
pickle.dump(text_configs,  open(os.path.join(MODEL_DIR, "model_configs.pkl"), "wb"))
pickle.dump(image_configs, open(os.path.join(MODEL_DIR, "imgae_model_configs.pkl"), "wb"))
print("restored detectors from", m["created"], "trained on", m["dataset"]["pcaps"])
print("Stage 3 / 4 can run now — Stages 1-2 not needed.")

## Stage 3 — LLM finetuning (QLoRA, GPU)

4-bit LoRA SFT of Llama-3-8b — plain **peft + bitsandbytes** (no unsloth) — to generate benign/malicious
traffic in the dataset's format. Fits a T4 (MAX_SEQ 1024, batch 1 + grad-accum). Uses TRL **`SFTConfig`**;
saves the adapter to `model/llama_lora`.

In [ ]:
import collections, os, torch
prepare()   # re-establish repo + config, so this stage runs standalone
# Stage 3 deps — plain peft + bitsandbytes QLoRA (NO unsloth); trl>=0.11 gives SFTConfig.
!pip -q install "trl>=0.11" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.34" "huggingface_hub"
assert torch.cuda.is_available(), "Stage 3 needs a GPU runtime"

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

MODEL   = "unsloth/llama-3-8b-bnb-4bit"
MAX_SEQ = 1024
_bf16   = torch.cuda.is_bf16_supported()

# Download and parse HF_LLM_PCAPS with the shared helpers (PCAP helpers cell): the first
# HF_LLM_MAX_REQUESTS unique HTTP requests, headers included, then tshark is stopped.
data = collect_requests(HF_LLM_PCAPS, HF_LLM_MAX_REQUESTS, full_scan=False)
if not data:
    raise RuntimeError("No HTTP requests found. Choose a PCAP containing plaintext HTTP traffic.")
print("LLM training requests:", len(data), dict(collections.Counter(r["Label"] for r in data)))

# Load the QLoRA base model only after the potentially large PCAP download/parse.
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", dtype=torch.float16)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def json_to_string(d, indent=0):
    out, pad = [], " " * indent
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, (dict, list)):
                out.append(f"{pad}{k}:"); out.append(json_to_string(v, indent + 1))
            else:
                out.append(f"{pad}{k}: {v}")
    elif isinstance(d, list):
        for it in d:
            out.append(json_to_string(it, indent))
    else:
        out.append(f"{pad}{d}")
    return "\n".join(out)

alpaca = ("Below is an instruction that describes a task, paired with an input that provides "
          "further context. Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}")
EOS = tok.eos_token
def body(it):
    return it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
texts = [alpaca.format(
            "Follow these tips to generate malicious http traffic" if it["Label"] == "Malicious"
            else "Follow these tips to generate benign http traffic",
            it["Request Line"], body(it)) + EOS
         for it in data]
ds = Dataset.from_dict({"text": texts}).shuffle(seed=42)

trainer = SFTTrainer(
    model=model, processing_class=tok, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ, packing=False,
        per_device_train_batch_size=1, gradient_accumulation_steps=16,
        warmup_steps=5, max_steps=60, learning_rate=2e-4,
        fp16=not _bf16, bf16=_bf16,
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=5, optim="paged_adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407,
        output_dir=os.path.join(MODEL_DIR, "llama_outputs"), report_to="none"))
trainer.train()
model.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
tok.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
print("saved LoRA ->", os.path.join(MODEL_DIR, "llama_lora"))

## Stage 4 — PPO adversarial generation (GPU)

Hand-rolled PPO in `RL-Adv/ppo_core.py` (**no trl-PPO** — immune to trl API churn) tunes a generator
to flip the frozen detectors' predictions. Reward = mean detector probability of the *opposite*
label. `policy="pythia"` is the cheap demo generator; **Stage 4b** runs the Stage-3 Llama (paper-faithful).

In [ ]:
import os, sys, importlib
prepare()   # re-establish repo + config, so this stage runs standalone
# deps already installed by Stage 2 (transformers, datasets); torch comes from the runtime.
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)                       # so ppo_core's ../model and ../dataset paths resolve
import ppo_core; importlib.reload(ppo_core)

# policy="pythia" -> cheap demo generator (EleutherAI/pythia-160m). FEATURE_TYPE: "Text" | "Image".
# Tune the knobs (steps / sample_size / batch_size) as needed; batch_size must be 1..4.
asr = ppo_core.train_ppo(policy="pythia", feature_type="Text",
                         steps=40, sample_size=2000, batch_size=4)
os.chdir(REPO)
print("Stage 4 (pythia) ASR:", asr)

In [ ]:
import os
for p in [os.path.join(REPO, "dataset", "train_data2.json"),
          os.path.join(REPO, "model", "model_configs.pkl"),
          os.path.join(REPO, "model", "llama_lora", "adapter_config.json")]:
    print(os.path.exists(p), p)

In [ ]:
import glob, json, os

files = sorted(glob.glob(os.path.join(REPO, "dataset", "PPO_data", "**", "*.json"), recursive=True))
if files:
    print("latest:", files[-1])
    print(json.dumps(json.load(open(files[-1]))[:2], indent=2)[:1500])
else:
    print("no PPO_data yet — run Stage 4 first")

## Stage 4b (faithful): PPO-tune the Stage-3 Llama instead of pythia-160m

In [ ]:
import torch
print(f"GPU used: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import os
lora = os.path.join(REPO, "model", "llama_lora")
ckpt = os.path.join(REPO, "model", "ppo_llama_ckpt")
print("Stage 3 LoRA present:", os.path.isfile(os.path.join(lora, "adapter_config.json")))
print("  contents:", os.listdir(lora) if os.path.isdir(lora) else "(missing)")
print("PPO checkpoint present:", os.path.isdir(ckpt) and os.path.exists(os.path.join(ckpt, "progress.json")))

In [ ]:
# ── Stage 4b (faithful): PPO-tune the Stage-3 Llama (policy="llama") ──
# Needs model/llama_lora from Stage 3 on disk. Checkpoints to model/ppo_llama_ckpt (auto-resume
# on re-run). Heavier than pythia — tiny defaults "prove it runs"; raise steps/sample_size and use
# a bigger GPU for paper-grade. Same hand-rolled loop as Stage 4, just policy="llama".
import os, sys, importlib
prepare()
# deps already installed by Stage 3 (peft + bitsandbytes), whose model/llama_lora this consumes.
RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)
import ppo_core; importlib.reload(ppo_core)
asr = ppo_core.train_ppo(policy="llama", feature_type="Text",
                         steps=15, sample_size=256, batch_size=2)
os.chdir(REPO)
print("Stage 4b (llama) ASR:", asr)